In [7]:
import re
import pandas as pd
from collections import Counter

In [5]:
df = pd.read_csv("../data/processed/listing_sample.csv")

In [6]:
df.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1169503734,133 Crystal Street,Taft,3.0,2.0,235000,This unique property offers two homes on one l...
1,1154220129,15664 Kadota,Victorville,4.0,4.0,459000,Beautiful Two-Story Home in Victorville – Spac...
2,1159921951,949 S Manhattan Place 203,Los Angeles,3.0,2.0,689000,Presenting this exquisite second-floor corner ...
3,1170333192,1872 Seascape Boulevard,Aptos,3.0,3.0,1195000,Thoughtfully renovated coastal retreat tucked ...
4,1152463169,2085 Westhampton Drive,Arroyo Grande,3.0,2.0,1050000,Welcome to 2085 Westhampton Drive in Arroyo Gr...


In [8]:
class TextCleaner: 
    def __init__(self): 
        self.abbrev_map = { 
        'br': 'bedroom', 'ba': 'bathroom', 'sqft': 'square feet', 
        'w/': 'with', 'w/o': 'without', 'mbr': 'master bedroom' 
    } 
    def clean_text(self, text): 
        text = self.normalize_unicode(text) 
        text = self.normalize_prices(text) 
        text = self.normalize_measurements(text) 
        text = self.expand_abbreviations(text) 
        return text.strip() 
    def normalize_prices(self, text): 
        # 450k → 450000 
        text = re.sub(r'(\d+)k', lambda m: str(int(m.group(1))*1000), text, 
        flags=re.I) 
        # 1.2m → 1200000 
        text = re.sub(r'(\d+\.?\d*)m', lambda m: 
        str(int(float(m.group(1))*1000000)), text, flags=re.I) 
        return text 
    def _extract_top_ngrams(self, series, n=20):
        words = []
        for text in series.dropna(): 
            words.extend(re.findall(r'\b\w+\b', text.lower()))
        return Counter(words).most_common(n)
    def _detect_abbreviations(self, series):
        counts = Counter()
        for text in series.dropna():
            words = re.findall(r'\b\w+\b', text.lower())
            for word in words:
                if word in self.abbrev_map:
                    counts[word] += 1
        return counts.most_common(10)

    def profile_column(self, df, column_name): 
        """Analyze what's actually in L_Remarks""" 
        return { 
            'null_rate': df[column_name].isnull().mean(), 
            'avg_length': df[column_name].str.len().mean(), 
            'common_terms': self._extract_top_ngrams(df[column_name]), 
            'price_mentions': df[column_name].str.contains(r'\$\d').sum(), 
            'has_html': df[column_name].str.contains('<').sum(), 
            'common_abbreviations': self._detect_abbreviations(df[column_name]) 
        } 

In [9]:
# Use this to guide your cleaning strategy: 
cleaner = TextCleaner()
df = pd.read_csv("../data/processed/listing_sample.csv")
profile = cleaner.profile_column(df, 'remarks') 
print(f"HTML tags found in {profile['has_html']} listings") 
print(f"Common abbreviations: {profile['common_abbreviations']}")

HTML tags found in 1 listings
Common abbreviations: [('sqft', 32), ('ba', 4), ('br', 3)]
